# Save and load a `.tlspec`

TorchLens 2.0 uses a unified `.tlspec` directory format for portable saved logs.

In [ ]:
from pathlib import Path
import importlib.util
import sys
from tempfile import TemporaryDirectory

import torch
import torchlens as tl

shared_path = None
for root in [Path.cwd(), *Path.cwd().parents]:
    candidate = root / "notebooks" / "total_audit" / "_shared.py"
    if candidate.exists():
        shared_path = candidate
        break
if shared_path is None:
    raise FileNotFoundError("Could not find notebooks/total_audit/_shared.py")

spec = importlib.util.spec_from_file_location("torchlens_total_audit_shared", shared_path)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load {shared_path}")
shared = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = shared
spec.loader.exec_module(shared)
tiny_model = shared.tiny_model

torch.manual_seed(0)
model = tiny_model(seed=7).eval()
x = torch.randn(2, 4)

Round-trip a log with the public `tl.save` and `tl.load` helpers.

In [ ]:
with TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "tiny_model_log.tlspec"
    log = tl.log_forward_pass(model, x, layers_to_save="all")
    tl.save(log, path, overwrite=True)
    loaded = tl.load(path)
    print(path.name)
    print(type(loaded).__name__)
    print(torch.allclose(log["relu_1_2"].activation, loaded["relu_1_2"].activation))

Portable bundles include activation sidecars when `include_activations=True`, which is the default.